In [2]:
from langchain_ollama import OllamaLLM 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

chain = (
    PromptTemplate.from_template(
        """Given the user question below, classify it as either being about `Films`, `Cars`, or `Other`.

Do not respond with more than one word.

<question>
{question}
</question>

Classification:"""
    )
    | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")
    | StrOutputParser()
)

chain.invoke({"question": "Tell me how to clean my ford focus properly?"})

'Cars'

In [65]:
# Sub chains
# tools_chain = PromptTemplate.from_template(

film_chat_chain = PromptTemplate.from_template(
    """You are a friendly and knowledgeable assistant specializing in films. Your goal is to engage users in conversations to understand their film preferences, interests, and what aspects of films are most important to them. \
    Keep the conversation a reasonably short length, and ask questions to get more information from the user. \
    
    Respond to the following question:
    
    Question: {question}
    Answer:
    """
    ) | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")

explain_rec_chain = PromptTemplate.from_template(
    """You are talking to the user, briefly explain the recommendation score breakdown to the user for the given film. 
    The recommendation score format is:
    `Title (Release Date): total_score, cast_score, director_score, genre_score, collaborative_filtering_score`. 
    total_score is how likely this film is to be recommended to the user.
    the other scores are the content based scores of the user with collaborative filtering score based on other uses reviews.
    Given the following breakdown:
    
    {recommendation_text}
    
    You need to concisely and briefly say to the user why the film might be recommended.
    
    Answer:
    """
    # """You are an expert in films, and you are talking to a user. You need to briefly go through as to why a particular film was recommended to the user. 
    # The recommendation score format is: 
    # `Title (Release Date): total_score, cast_score, director_score, genre_score, collaborative_filtering_score`. 
    # Given the following breakdown:
    # 
    # {input_text}
    # 
    # Provide a clear and concise explanation of the scores and why the film might be recommended.
    # 
    # Answer:
    # """
    ) | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")

glean_feed_back_chain = PromptTemplate.from_template(
    """
    Analyze the sentiment of the following sentence.
    
    Provide sentiment scores on the importance of the following features followed with the sentiment scores for items within these features:
    - Films
    - Actors
    - Directors
    - Genres
    Add a `\n` before each feature, and a `,` between each item rating.
    
    Don't be verbose, just provide the scores with no other info.
    - For item ratings, just write i: score, e.g. "item1: 0.5".
    - For feature importance, just write f; score, e.g. "feature1; 0.5".
    For example:
    \n Films; 0.5 item1: 0.9, item2: 0.2 \n Actors; 0.3 item1: 0.5, item2: 0.1
    
    Sentence: {input_text}
    """
    ) | OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")


In [61]:
explain_rec_chain.invoke({"input_text": "Batman Begins,2005-06-10,12.571,5.40,4.88,7.4620,3.286"})

'The film "Batman Begins" (2005) has a total score of 12.57, indicating that it\'s likely to be recommended to you. The high scores in cast (5.40), director (4.88), and collaborative filtering (7.46) suggest that the film is well-liked by experts and users alike, making it a strong recommendation based on content similarity.'

In [8]:
film_chat_chain.invoke({"question": "Hey, I like the movie 'The Dark Knight', I'm in between minds about what really defined it as a great movie."})

'"The Dark Knight" is an incredible film that has left a lasting impact on the superhero genre. What specifically do you think was missing or didn\'t quite come together for you to consider it truly great? Was it Heath Ledger\'s iconic performance, the themes of chaos and anarchy, or perhaps the way Christopher Nolan balanced action and drama?\n\nAlso, are you more interested in the technical aspects of filmmaking (e.g., cinematography, score), the performances, the storytelling, or something else that contributed to your impression of the movie?'

In [59]:
explain_rec_chain.invoke({"input_text": "Batman Begins,2005-06-10,12.571,5.40,4.88,7.4620,3.286"})

"The movie recommendation score breakdown is:\n\n* Total Score: 12.57 (indicating a high likelihood of recommendation)\n* Cast Score: 5.40 (suggesting that the cast may not be a strong factor in recommending this film to you)\n* Director Score: 4.88 (implying that the director's style or expertise may influence your interest in this movie)\n* Genre Score: 7.46 (indicating that the genre might align with your tastes and interests)\n* Collaborative Filtering Score: 3.29 (suggesting that reviews from other users may have helped recommend this film to you)\n\nBased on this breakdown, Batman Begins is likely recommended because its genre score suggests it may appeal to you, while its cast score is not as high."

In [58]:
glean_feed_back_chain.invoke({"input_text": "I like the film bullet train. In my opinion, directors are more important in films, I don't think actors are important. I don't mind what genre it is tbh. I do think Paul Mescal is a good actor though."})

'*Films*: 0.8 **\n*Actors*: 0.6 **\n*Directors*: 0.7 **\n*Genres*: -0.2 **\n\n"bullet train": 0.9\n""average film rating"": 0.4\n"Paul Mescal": 0.8'

In [66]:
glean_feed_back_chain.invoke({"input_text": "I like the film bullet train. In my opinion, directors are more important in films, I don't think actors are important. I don't mind what genre it is tbh. I do think Paul Mescal is a good actor though."})

'`Films; 0.6, item1: 0.9`\n`Directors; 0.8`\n`Actors; 0.2, item1: 0.5, item2: 0.1`\n`Genres; 0.4`'

In [64]:
glean_feed_back_chain.invoke({"input_text": "I like the film bullet train. In my opinion, directors are more important in films, I don't think actors are important. I don't mind what genre it is tbh. I do think Paul Mescal is a good actor though."})

'**Films: 0.6**\n "Films: 0.8"\n "Films: 0.2"\n\n**Actors: -0.4**\n"Actors: 0.5"\n"Actors: -0.7"\n\n**Directors: 0.9**\n "Directors: 0.8"\n "Directors: 0.1"\n\n**Genres: -0.1**\n"Genres: 0.2"\n"Genres: -0.3"'

In [46]:
glean_feed_back_chain.invoke({"input_text": "I like the film bullet train. In my opinion, directors are 'more important in films, I don't think actors are important. I don't mind what genre it is tbh. I do think Paul Mescal is a good actor though. I'm not sure where I sit on my opinion of the Hunger Games, I think it's a bit overrated. I do like the film The Dark Knight though, it's a classic. Also, I believe 'the music composer can make or break the film."})

"**Feature Sentiment Scores**\n- Films | Positive\n- Actors | Neutral\n- Directors | Negative\n- Genres | Neutral\n\n**Item Ratings and Feature Importance**\n- Films | The Dark Knight (Positive) - 9.5, |Films| (Positive)\n- Films | Bullet Train (Negative) - 6.5, |Films|\n- Films | Hunger Games (Negative) - 7.8, |Films|\n- Actors | Paul Mescal (Positive) - 8.2, |Actors|\n- Directors | Not mentioned directly, but implied as 'more important' than actors in films, |Directors| (Neutral)\n- Genres | TBH (Neutral), no specific genre mentioned, |Genres| (Neutral)\n- Music Composer | Not explicitly mentioned, but implied as a crucial element, Music Composer (Positive)"

In [ ]:

    - Item ratings; item ratings within a feature, e.g. scores of various films. Item and score surrounded by quotes "".
    - Feature importance; importance of the feature to the user. Feature and score surrounded by asterisks **.

In [ ]:
def regex_item_score(text):
    import re
    return re.findall(r'"(.*?)"', text)

def regex_feature_score(text):
    import re
    return re.findall(r'\*\*(.*?)\*\*', text)

def glean_feed_back_chain(input_text):
    feature_scores = regex_feature_score(input_text)
    item_scores = regex_item_score(input_text)
    change_temp_user_profile(feature_scores, item_scores)
    return feature_scores, item_scores



In [ ]:
def change_temp_user_profile(feature_scores, item_scores):
    for feature in feature_scores:
        print(feature)
    for item in item_scores:
        print(item)

In [35]:
def route(info):
    if ""
    film_chat_chain.invoke({"question": info["message"]})
    glean_feed_back_chain.invoke({"input_text": info["message"]})
    return general_chain
    
    
    # if "anthropic" in info["topic"].lower():
    #     return anthropic_chain
    # elif "langchain" in info["topic"].lower():
    #     return langchain_chain
    # else:
    #     return general_chain
